# Insert Records via Debezium CDC + PostgreSQL
---
**DADS6005 Quiz 2 — KsqlDB**

1. เชื่อมต่อ PostgreSQL
2. อ่านข้อมูลจาก CSV
3. INSERT ทีละแถวเพื่อจำลอง CDC (Change Data Capture)
4. Debezium จะจับการเปลี่ยนแปลงและส่งเข้า Kafka

In [ ]:
import psycopg2
import pandas as pd
import time

# Connect to PostgreSQL
conn = psycopg2.connect(
    database="dbfood",
    host="localhost",
    user="postgres",
    password="123456",
    port="5433"
)
cur = conn.cursor()
print("[OK] Connected to PostgreSQL")

In [ ]:
# Load CSV data
df = pd.read_csv("food_coded.csv")
print(f"[OK] Loaded {len(df)} records from food_coded.csv")
df.head(2)

In [ ]:
# Create target table if not exists
cur.execute("""
CREATE TABLE IF NOT EXISTS db_food_coded2_stream (
    ids INTEGER,
    gpa TEXT,
    gender INTEGER,
    breakfast INTEGER,
    calories_chicken INTEGER,
    calories_day INTEGER,
    calories_scone INTEGER,
    coffee INTEGER,
    comfort_food TEXT,
    comfort_food_reasons TEXT,
    cook INTEGER,
    diet_current TEXT,
    diet_current_coded INTEGER,
    eating_out INTEGER,
    exercise INTEGER,
    fav_cuisine TEXT,
    fruit_day INTEGER,
    income INTEGER,
    sports INTEGER,
    thai_food INTEGER,
    veggies_day INTEGER,
    vitamins INTEGER,
    waffle_calories INTEGER,
    tortilla_calories INTEGER,
    turkey_calories INTEGER,
    weights TEXT,
    PRIMARY KEY (ids)
);
""")
conn.commit()
print("[OK] Table ready")

---
## Stream Data — INSERT ทีละแถว (Simulate CDC)

Debezium จะ capture ทุก INSERT และส่งไปยัง Kafka topic `source.public.db_food_coded2_stream`

In [ ]:
# Truncate before inserting
cur.execute("TRUNCATE TABLE db_food_coded2_stream;")
conn.commit()

inserted = 0
for _, row in df.iterrows():
    cur.execute("""
        INSERT INTO db_food_coded2_stream (
            ids, gpa, gender, breakfast, calories_chicken, calories_day,
            calories_scone, coffee, comfort_food, comfort_food_reasons,
            cook, diet_current, diet_current_coded, eating_out, exercise,
            fav_cuisine, fruit_day, income, sports, thai_food,
            veggies_day, vitamins, waffle_calories, tortilla_calories,
            turkey_calories, weights
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        int(row['ids']), str(row['gpa']), int(row['gender']),
        int(row['breakfast']), int(row['calories_chicken']), int(row['calories_day']),
        int(row['calories_scone']), int(row['coffee']), str(row['comfort_food']),
        str(row['comfort_food_reasons']), int(row['cook']), str(row['diet_current']),
        int(row['diet_current_coded']), int(row['eating_out']), int(row['exercise']),
        str(row['fav_cuisine']), int(row['fruit_day']), int(row['income']),
        int(row['sports']), int(row['thai_food']), int(row['veggies_day']),
        int(row['vitamins']), int(row['waffle_calories']),
        int(row['tortilla_calories']), int(row['turkey_calories']),
        str(row['weights'])
    ))
    conn.commit()
    inserted += 1
    print(f"[{inserted}] Inserted id={row['ids']} | comfort_food_reasons={str(row['comfort_food_reasons'])[:30]}...")
    time.sleep(1)

cur.close()
conn.close()
print(f"\n[Done] Inserted {inserted} records — CDC events sent to Kafka!")